In [1]:
from collections import Counter

# -------------------------------
# Step 1 & 2: Read the Corpus
# -------------------------------
with open("corpus.txt", "r") as file:
    words = file.read().splitlines()

# -------------------------------
# Step 3: Preprocess the Text
# -------------------------------
words = [word.lower().strip() for word in words if word.strip()]

# -------------------------------
# Step 4 & 5:
# Split into characters and
# Count word frequencies
# -------------------------------
vocab = Counter()

for word in words:
    token = " ".join(list(word)) + " </w>"
    vocab[token] += 1

print("Initial Vocabulary\n")

for word, freq in vocab.items():
    print(f"{word} : {freq}")

# -------------------------------
# Step 6 & 7:
# Count adjacent symbol pairs
# -------------------------------
def count_pairs(vocabulary):

    pairs = Counter()

    for word, freq in vocabulary.items():

        symbols = word.split()

        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq

    return pairs


# -------------------------------
# Step 9 & 10:
# Merge the selected pair
# -------------------------------
def merge_pair(pair, vocabulary):

    first, second = pair
    new_vocab = {}

    for word, freq in vocabulary.items():

        symbols = word.split()

        merged = []

        i = 0

        while i < len(symbols):

            if (
                i < len(symbols)-1
                and symbols[i] == first
                and symbols[i+1] == second
            ):

                merged.append(first + second)
                i += 2

            else:

                merged.append(symbols[i])
                i += 1

        new_vocab[" ".join(merged)] = freq

    return Counter(new_vocab)


# -------------------------------
# Step 11 & 12:
# Repeat merges and store rules
# -------------------------------
merge_rules = []

num_merges = 10

for step in range(num_merges):

    print("\n==============================")
    print("Merge", step + 1)

    pairs = count_pairs(vocab)

    if len(pairs) == 0:
        break

    best_pair = pairs.most_common(1)[0][0]

    print("Most Frequent Pair:", best_pair)
    print("Frequency:", pairs[best_pair])

    merge_rules.append(best_pair)

    vocab = merge_pair(best_pair, vocab)

    print("\nVocabulary After Merge")

    for word in vocab:
        print(word)

# -------------------------------
# Step 13:
# Build Final Vocabulary
# -------------------------------
final_vocab = set()

for word in vocab:
    final_vocab.update(word.split())

# -------------------------------
# Step 14:
# Encoder
# -------------------------------
def encode(word, merge_rules):

    tokens = list(word.lower()) + ["</w>"]

    for first, second in merge_rules:

        merged = []

        i = 0

        while i < len(tokens):

            if (
                i < len(tokens)-1
                and tokens[i] == first
                and tokens[i+1] == second
            ):

                merged.append(first + second)
                i += 2

            else:

                merged.append(tokens[i])
                i += 1

        tokens = merged

    return tokens


# -------------------------------
# Step 15:
# Decoder
# -------------------------------
def decode(tokens):

    word = "".join(tokens)

    return word.replace("</w>", "")


# -------------------------------
# Step 16:
# Test the Tokenizer
# -------------------------------
test_word = "playing"

encoded = encode(test_word, merge_rules)

decoded = decode(encoded)

# -------------------------------
# Step 17:
# Display Results
# -------------------------------
print("\n==============================")
print("Merge Rules")

for i, rule in enumerate(merge_rules, start=1):
    print(f"{i}. {rule}")

print("\nFinal Vocabulary")

print(sorted(final_vocab))

print("\nVocabulary Size:", len(final_vocab))

print("\nTest Word")

print("Original :", test_word)

print("Encoded  :", encoded)

print("Decoded  :", decoded)

Initial Vocabulary

l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
n e w e s t </w> : 1

Merge 1
Most Frequent Pair: ('w', 'e')
Frequency: 4

Vocabulary After Merge
l o w </w>
l o we r </w>
l o we s t </w>
n e w </w>
n e we r </w>
n e we s t </w>

Merge 2
Most Frequent Pair: ('l', 'o')
Frequency: 3

Vocabulary After Merge
lo w </w>
lo we r </w>
lo we s t </w>
n e w </w>
n e we r </w>
n e we s t </w>

Merge 3
Most Frequent Pair: ('n', 'e')
Frequency: 3

Vocabulary After Merge
lo w </w>
lo we r </w>
lo we s t </w>
ne w </w>
ne we r </w>
ne we s t </w>

Merge 4
Most Frequent Pair: ('w', '</w>')
Frequency: 2

Vocabulary After Merge
lo w</w>
lo we r </w>
lo we s t </w>
ne w</w>
ne we r </w>
ne we s t </w>

Merge 5
Most Frequent Pair: ('lo', 'we')
Frequency: 2

Vocabulary After Merge
lo w</w>
lowe r </w>
lowe s t </w>
ne w</w>
ne we r </w>
ne we s t </w>

Merge 6
Most Frequent Pair: ('r', '</w>')
Frequency: 2

Vocabulary After Merge
lo w</w>
lowe r</

In [ ]:
from collections import defaultdict

corpus = [
    "i love you",
    "i love python",
    "i love coding",
    "how are you",
    "how are they",
    "good morning everyone",
    "good morning teacher"
]

model = defaultdict(list)

# Train the model
for sentence in corpus:
    words = sentence.lower().split()

    for i in range(len(words)-1):
        model[words[i]].append(words[i+1])

# Predict next word
def predict(text):

    last_word = text.lower().split()[-1]

    if last_word in model:

        words = model[last_word]

        # Return the most common next word
        return max(set(words), key=words.count)

    return "No prediction"

# Test
while True:

    sentence = input("Enter text: ")

    if sentence.lower() == "exit":
        break

    print("Next Word:", predict(sentence))

Enter text:  i love


Next Word: coding


Enter text:  how are


Next Word: they


Enter text:  good morning


Next Word: teacher


Enter text:  my name


Next Word: No prediction
